# E5 — Procedural descriptor round-trip

**Goal.** Test whether the PhenoSuite procedural model — parameterised by just three numbers per leaf (`leaf_angle`, `droopiness`, `leaf_length`) — can faithfully represent real sorghum plant architecture extracted from 3D voxel skeletons.

**Why this matters.** The procedural representation is the compressed, generative form of the descriptor. If we can fit procedural parameters from real skeletons and recover biologically meaningful traits from the fitted descriptor, then:

- The procedural model is expressive enough for real plants (E4 geometric fidelity becomes tractable).
- Population-level analysis of the *parameters themselves* is meaningful (E8 heritability of descriptor params).
- Synthetic plants drawn from fitted parameter distributions are realistic (E9 distribution match).

**Pipeline (per plant):**
1. Load skeleton voxels → segment stem + leaves → rotate to Y-up.
2. Per leaf: build world-space spline → project into leaf-local frame.
3. **Inverse fit**: optimise `leaf_angle` and `droopiness` to minimise shape error between the observed local spline and the procedural forward model (`_build_leaf_center_local`). `leaf_length` is set from arc length (exact).
4. Write a procedural descriptor XML (`useCtrlOverrides=0`, `leafAzimuthDeg` per leaf).
5. Read the descriptor back → extract traits via PhenoSuite's standard pipeline → compare θ and φ with the gold standard (`angles.txt`).

**Comparison baseline.** Notebook `03_sec2.2_phenosuite_traits_vs_gold.ipynb` (E1b) performs the same comparison but using spline-override descriptors (`useCtrlOverrides=1`), which encode the skeleton shape exactly. The gap between E1b and E5 tells us how much information is lost by compressing to three procedural parameters.

**Input.** 332 sorghum plants with usable skeletons (2018 SAP, Zenodo voxel dataset).

**Output.** Per-leaf fitted parameters, fit residuals, trait round-trip agreement vs gold, parameter distributions, and comparison with E1b.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1] if Path.cwd().name == '01_phyllotaxis' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

from experiments import paths, data_loaders as dl
from phenosuite.skeleton_to_descriptor import (
    fit_skeleton_procedural,
    skeleton_to_leaf_splines,
    extract_traits_via_phenosuite,
)
from phenosuite.traits import compute_traits_from_descriptor

paths.assert_data_present()

FIG_DIR = Path('figures'); FIG_DIR.mkdir(exist_ok=True)
OUT_DIR = Path('outputs'); OUT_DIR.mkdir(exist_ok=True)

SPLINE_POINTS = 8     # procedural control points for the forward model
STEM_RADIUS   = 0.015 # metres — typical sorghum stem radius

## 1. Fit procedural parameters on every plant

For each of the 332 skeletons, segment and fit `leaf_angle`, `droopiness`, and `leaf_length` per leaf. Also record the extracted `azimuth_deg` and inter-node `distance`. About 2 minutes total.

In [ ]:
plants = dl.list_voxel_plants(require_skeleton=True)
print(f'Fitting procedural parameters for {len(plants)} plants (spline_points={SPLINE_POINTS})...')

all_fits = []
failures = []
for i, plant_id in enumerate(plants):
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(plants)}')
    try:
        sk = dl.load_skeleton(plant_id)
        df_fit = fit_skeleton_procedural(
            sk,
            stem_radius=STEM_RADIUS,
            spline_points=SPLINE_POINTS,
        )
        df_fit['plant_id'] = plant_id
        all_fits.append(df_fit)
    except Exception as e:
        failures.append((plant_id, type(e).__name__, str(e)))

fits = pd.concat(all_fits, ignore_index=True)
print(f'\nDone. {len(fits)} leaves across {fits["plant_id"].nunique()} plants. Failures: {len(failures)}.')
fits.head()

## 2. Fit quality — how well does the procedural model capture each leaf's shape?

The `residual_per_point` is the mean squared distance (in metres²) between the observed local-frame spline and the best-fit procedural spline, averaged over 50 arc-length-uniform sample points. Lower is better. For context, at typical leaf lengths of 0.2–0.4 m, a residual per point of 1e-4 m² corresponds to ~1 cm RMS positional error along the midrib.

In [ ]:
fits['rms_error_cm'] = np.sqrt(fits['residual_per_point']) * 100  # convert to cm

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Histogram of RMS error
ax = axes[0]
ax.hist(fits['rms_error_cm'], bins=50, color='#3a8c5e', alpha=0.85, edgecolor='white')
ax.axvline(fits['rms_error_cm'].median(), color='black', ls='--', lw=1.2,
           label=f'median = {fits["rms_error_cm"].median():.2f} cm')
ax.set_xlabel('RMS spline error (cm)')
ax.set_ylabel('Count')
ax.set_title('Fit residual per leaf')
ax.legend(fontsize=9)

# Residual vs leaf length
ax = axes[1]
ax.scatter(fits['leaf_length'] * 100, fits['rms_error_cm'], s=6, alpha=0.3, color='#3672b5')
ax.set_xlabel('Leaf length (cm)')
ax.set_ylabel('RMS spline error (cm)')
ax.set_title('Residual vs leaf length')

# Residual by node index
ax = axes[2]
node_stats = fits.groupby('leaf_index')['rms_error_cm'].agg(['median', 'mean', 'count'])
ax.bar(node_stats.index, node_stats['median'], color='#c25b56', alpha=0.85, edgecolor='white')
ax.set_xlabel('Leaf node (bottom → top)')
ax.set_ylabel('Median RMS error (cm)')
ax.set_title('Fit quality by canopy position')

fig.suptitle(f'E5 — Procedural fit quality, N = {len(fits)} leaves')
plt.tight_layout()
plt.savefig(FIG_DIR / 'e5_fit_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'RMS spline error summary:')
print(f'  Median: {fits["rms_error_cm"].median():.2f} cm')
print(f'  Mean:   {fits["rms_error_cm"].mean():.2f} cm')
print(f'  90th %: {fits["rms_error_cm"].quantile(0.9):.2f} cm')
print(f'  Max:    {fits["rms_error_cm"].max():.2f} cm')

## 3. Fitted parameter distributions

What does the sorghum population look like in procedural-parameter space? Biologically, `leaf_angle` is the insertion angle (higher = more erect), `droopiness` is gravity droop (negative = tip hangs down), and `leaf_length` is the midrib arc length.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
cmap = plt.get_cmap('viridis')
max_node = min(9, int(fits['leaf_index'].max()))

# Top row: marginal distributions
for ax, col, label, color in [
    (axes[0, 0], 'leaf_angle',  'Fitted leaf_angle (deg)', '#c25b56'),
    (axes[0, 1], 'droopiness',  'Fitted droopiness',       '#3672b5'),
    (axes[0, 2], 'leaf_length', 'Fitted leaf_length (m)',   '#3a8c5e'),
]:
    ax.hist(fits[col], bins=50, color=color, alpha=0.85, edgecolor='white')
    ax.axvline(fits[col].median(), color='black', ls='--', lw=1,
               label=f'median = {fits[col].median():.2f}')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

# Bottom row: parameters vs node index (boxplots)
for ax, col, label in [
    (axes[1, 0], 'leaf_angle',  'leaf_angle (deg)'),
    (axes[1, 1], 'droopiness',  'droopiness'),
    (axes[1, 2], 'leaf_length', 'leaf_length (m)'),
]:
    nodes = sorted(fits['leaf_index'].unique())
    data = [fits[fits['leaf_index'] == n][col].dropna().values for n in nodes]
    bp = ax.boxplot(data, positions=nodes, widths=0.6, patch_artist=True,
                    medianprops=dict(color='black', lw=1.5),
                    flierprops=dict(marker='.', markersize=2, alpha=0.3))
    for patch, n in zip(bp['boxes'], nodes):
        patch.set_facecolor(cmap(n / max(max_node, 1)))
        patch.set_alpha(0.7)
    ax.set_xlabel('Leaf node (bottom → top)')
    ax.set_ylabel(label)

fig.suptitle(f'Fitted procedural parameters, N = {len(fits)} leaves across {fits["plant_id"].nunique()} plants')
plt.tight_layout()
plt.savefig(FIG_DIR / 'e5_fitted_parameter_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Trait round-trip — procedural descriptor vs gold standard

Write a procedural descriptor XML for each plant, read it back through PhenoSuite's trait pipeline, and compare the resulting θ and φ against the gold standard from `angles.txt`. This is the core E5 result.

In [ ]:
import tempfile, os
from phenosuite.skeleton_to_descriptor import skeleton_to_procedural_xml

def wrap180(x):
    return ((np.asarray(x) + 180) % 360) - 180

print(f'Running trait round-trip for {fits["plant_id"].nunique()} plants...')

records = []
rt_failures = []
with tempfile.TemporaryDirectory() as tmpdir:
    for i, plant_id in enumerate(fits['plant_id'].unique()):
        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{fits["plant_id"].nunique()}')
        try:
            # Load gold standard
            gold = dl.load_angles_txt(plant_id)
            sk = dl.load_skeleton(plant_id)

            # Write procedural descriptor XML
            xml_path = Path(tmpdir) / f'{plant_id}.xml'
            skeleton_to_procedural_xml(
                sk, xml_path,
                stem_radius=STEM_RADIUS,
                spline_points=SPLINE_POINTS,
            )

            # Read back and extract traits via PhenoSuite
            trait_list = compute_traits_from_descriptor(xml_path)

            # Match leaves bottom-to-top (same ordering as gold)
            n = min(len(gold), len(trait_list))
            for li in range(n):
                t = trait_list[li]
                azimuth = t['leaf_angle']['azimuth_deg']
                inclination = t['leaf_angle']['inclination_deg']
                theta_rt = 90.0 - inclination
                phi_rt = ((180.0 - azimuth) % 360.0) - 180.0

                records.append({
                    'plant_id': plant_id,
                    'leaf_index': li,
                    'theta_gold': gold.iloc[li]['theta'],
                    'phi_gold': gold.iloc[li]['phi'],
                    'theta_rt': theta_rt,
                    'phi_rt': phi_rt,
                    'leaf_length_rt': t['leaf_length'],
                    'azimuth_rt': azimuth,
                    'inclination_rt': inclination,
                })
        except Exception as e:
            rt_failures.append((plant_id, type(e).__name__, str(e)))

rt = pd.DataFrame(records).dropna(subset=['theta_rt', 'phi_rt']).copy()
print(f'\nDone. {len(rt)} leaf comparisons across {rt["plant_id"].nunique()} plants. Failures: {len(rt_failures)}.')

# Per-plant residual φ-sign alignment (same approach as E1b)
flipped = set()
for pid, grp in rt.groupby('plant_id'):
    g, o = grp['phi_gold'].values, grp['phi_rt'].values
    if (wrap180(o + 180 - g) ** 2).sum() < (wrap180(o - g) ** 2).sum():
        flipped.add(pid)

mask = rt['plant_id'].isin(flipped)
rt.loc[mask, 'phi_rt'] = wrap180(rt.loc[mask, 'phi_rt'].values + 180)

rt['dtheta'] = rt['theta_rt'] - rt['theta_gold']
rt['dphi']   = wrap180(rt['phi_rt'].values - rt['phi_gold'].values)
rt['abs_dtheta'] = rt['dtheta'].abs()
rt['abs_dphi']   = rt['dphi'].abs()

print(f'Plants with +180° φ flip: {len(flipped)} / {rt["plant_id"].nunique()}')

In [ ]:
def report(label, sub):
    print(f'\n=== {label} (N={len(sub)}) ===')
    if len(sub) < 2:
        print('  (too few)'); return
    rt_theta = pearsonr(sub['theta_gold'], sub['theta_rt'])[0]
    rp_phi   = pearsonr(sub['phi_gold'],   sub['phi_rt'])[0]
    print(f'  Median |Δθ|: {sub["abs_dtheta"].median():.2f}°    Mean: {sub["abs_dtheta"].mean():.2f}°    RMSE: {np.sqrt((sub["dtheta"]**2).mean()):.2f}°')
    print(f'  Median |Δφ|: {sub["abs_dphi"].median():.2f}°    Mean: {sub["abs_dphi"].mean():.2f}°    RMSE: {np.sqrt((sub["dphi"]**2).mean()):.2f}°')
    print(f'  Pearson r:    θ={rt_theta:.3f}    φ={rp_phi:.3f}')
    print(f'  R²:           θ={rt_theta**2:.3f}    φ={rp_phi**2:.3f}')
    print(f'  Within 5°:    θ={(sub["abs_dtheta"]<5).mean()*100:.1f}%    φ={(sub["abs_dphi"]<5).mean()*100:.1f}%')
    print(f'  Within 10°:   θ={(sub["abs_dtheta"]<10).mean()*100:.1f}%    φ={(sub["abs_dphi"]<10).mean()*100:.1f}%')

lower4_rt = rt[rt['leaf_index'] < 4]
report('E5 round-trip — lower 4 leaves, all', lower4_rt)
report('E5 round-trip — all leaves', rt)

## 5. Scatter and Bland-Altman — procedural round-trip vs gold

The headline figure: does the procedural model preserve the biologically meaningful traits (θ, φ) after compressing each leaf to just three numbers?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cmap = plt.get_cmap('viridis')
max_node = int(rt['leaf_index'].max())

for ax, gold_col, rt_col, label, ax_range in [
    (axes[0], 'theta_gold', 'theta_rt', 'θ (insertion angle, °)', (0, 90)),
    (axes[1], 'phi_gold',   'phi_rt',   'φ (azimuth, °)',          (-180, 180)),
]:
    for node in sorted(rt['leaf_index'].unique()):
        sub = rt[rt['leaf_index'] == node]
        ax.scatter(sub[gold_col], sub[rt_col], s=8, alpha=0.45,
                   color=cmap(node / max(max_node, 1)),
                   label=f'leaf {node}' if node < 7 else None)
    ax.plot(ax_range, ax_range, 'k--', lw=0.7, alpha=0.6)
    ax.set_xlim(ax_range); ax.set_ylim(ax_range)
    ax.set_xlabel(f'Gold (Gaillard pipeline): {label}')
    ax.set_ylabel(f'E5 procedural round-trip: {label}')
    ax.set_aspect('equal')

axes[0].set_title('θ — leaf insertion angle')
axes[1].set_title('φ — leaf azimuth')
axes[0].legend(loc='upper left', frameon=True, fontsize=8)
fig.suptitle(f'E5 — Procedural round-trip vs gold, N = {len(rt)} leaves across {rt["plant_id"].nunique()} plants')
plt.tight_layout()
plt.savefig(FIG_DIR / 'e5_scatter_roundtrip_vs_gold.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
n_nodes = min(8, int(rt['leaf_index'].max()) + 1)
fig, axes = plt.subplots(2, n_nodes, figsize=(2.2 * n_nodes, 5), sharey=True)
for node in range(n_nodes):
    sub = rt[rt['leaf_index'] == node]
    ax_t, ax_p = axes[0, node], axes[1, node]
    ax_t.hist(sub['dtheta'], bins=40, color='#c25b56', alpha=0.85, edgecolor='white')
    ax_t.axvline(0, color='black', lw=0.6); ax_t.set_xlim(-60, 60)
    ax_t.set_title(f'leaf {node}\nN={len(sub)}', fontsize=9)
    ax_p.hist(sub['dphi'], bins=40, color='#3672b5', alpha=0.85, edgecolor='white')
    ax_p.axvline(0, color='black', lw=0.6); ax_p.set_xlim(-180, 180)
    if node == 0:
        ax_t.set_ylabel('Δθ count')
        ax_p.set_ylabel('Δφ count')
    ax_p.set_xlabel('Δφ (°)')

fig.suptitle('Fitted skeletons Δθ (top) and Δφ (bottom)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'e5_fitted_skeletons_delta_by_node.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Comparison with E1b baseline — how much does procedural compression cost?

E1b (`03_sec2.2_phenosuite_traits_vs_gold.ipynb`) encodes skeletons via spline overrides (`useCtrlOverrides=1`), which pass the skeleton shape through to the trait extractor exactly. E5 compresses each leaf to just three procedural parameters. The gap tells us the cost of that compression.

In [ ]:
# Load E1b results if available
e1b_path = OUT_DIR / 'e1b_per_leaf_phenosuite_vs_gold.csv'
if not e1b_path.exists():
    e1b_path = Path('outputs') / 'e1b_per_leaf_phenosuite_vs_gold.csv'

if e1b_path.exists():
    e1b = pd.read_csv(e1b_path)
    print(f'Loaded E1b baseline: {len(e1b)} leaf comparisons')

    # Build comparison table
    rows_cmp = []
    for label, sub_name, sub_e5, sub_e1b in [
        ('Lower 4', 'lower4',
         rt[rt['leaf_index'] < 4],
         e1b[e1b['leaf_index'] < 4]),
        ('All leaves', 'all', rt, e1b),
    ]:
        for experiment, sub, theta_col, phi_col, dtheta_col, dphi_col in [
            ('E5 (procedural)', sub_e5, 'theta_rt', 'phi_rt', 'dtheta', 'dphi'),
            ('E1b (spline override)', sub_e1b, 'theta_ours', 'phi_ours', 'dtheta', 'dphi'),
        ]:
            n = len(sub)
            if n < 2:
                continue
            r_t = pearsonr(sub['theta_gold'], sub[theta_col])[0]
            r_p = pearsonr(sub['phi_gold'], sub[phi_col])[0]
            rows_cmp.append({
                'Subset': label,
                'Experiment': experiment,
                'N': n,
                'Median |Δθ|': sub[f'abs_{dtheta_col}' if f'abs_{dtheta_col}' in sub.columns else 'abs_dtheta'].median(),
                'RMSE θ': np.sqrt((sub[dtheta_col] ** 2).mean()),
                'r(θ)': r_t,
                'R²(θ)': r_t ** 2,
                'Median |Δφ|': sub[f'abs_{dphi_col}' if f'abs_{dphi_col}' in sub.columns else 'abs_dphi'].median(),
                'RMSE φ': np.sqrt((sub[dphi_col] ** 2).mean()),
                'r(φ)': r_p,
                'R²(φ)': r_p ** 2,
            })

    cmp = pd.DataFrame(rows_cmp)
    print('\n' + cmp.to_string(index=False, float_format='{:.2f}'.format))
else:
    print('E1b output not found — run 03_sec2.2_phenosuite_traits_vs_gold.ipynb first to enable comparison.')
    print('Skipping comparison; E5 results stand alone.')

## 6b. Filtered figures — excluding outlier fits

Not every skeleton yields a clean procedural fit. Short leaves with < 20 voxels are noisy, and a small fraction of fits have large residuals (the procedural model can't represent S-shaped or curled leaves). We apply two filters:

1. **Voxel count ≥ 20** — drops tiny leaves where the spline is dominated by voxel-level noise.
2. **RMS fit error ≤ 90th percentile** — drops the worst ~10% of fits where the procedural model is a poor approximation.

All points are shown in the scatter, but **filtered-out points are grayed**. Headline numbers are reported on the clean subset only.

In [ ]:
# Merge fit residual + voxel count into round-trip dataframe
rt_merged = rt.merge(
    fits[['plant_id', 'leaf_index', 'rms_error_cm', 'leaf_voxel_count']],
    on=['plant_id', 'leaf_index'],
    how='left',
)

RMS_CUTOFF = rt_merged['rms_error_cm'].quantile(0.9)
VOXEL_CUTOFF = 20

rt_merged['keep'] = (rt_merged['rms_error_cm'] <= RMS_CUTOFF) & (rt_merged['leaf_voxel_count'] >= VOXEL_CUTOFF)
rt_clean = rt_merged[rt_merged['keep']].copy()

print(f'Filter: rms_error_cm ≤ {RMS_CUTOFF:.2f} cm (90th pct) AND leaf_voxel_count ≥ {VOXEL_CUTOFF}')
print(f'Kept: {len(rt_clean)} / {len(rt_merged)} leaves ({100*len(rt_clean)/len(rt_merged):.1f}%)')
print()

# Report on filtered subset
lower4_clean = rt_clean[rt_clean['leaf_index'] < 4]
report('E5 FILTERED — lower 4 leaves', lower4_clean)
report('E5 FILTERED — all leaves', rt_clean)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cmap = plt.get_cmap('viridis')
max_node = int(rt_merged['leaf_index'].max())

for ax, gold_col, rt_col, label, ax_range in [
    (axes[0], 'theta_gold', 'theta_rt', 'θ (insertion angle, °)', (0, 90)),
    (axes[1], 'phi_gold',   'phi_rt',   'φ (azimuth, °)',          (-180, 180)),
]:
    # Gray background: filtered-out points
    outliers = rt_merged[~rt_merged['keep']]
    ax.scatter(outliers[gold_col], outliers[rt_col], s=6, alpha=0.2,
               color='#bbbbbb', zorder=1, label='filtered out')
    # Colored foreground: kept points
    for node in sorted(rt_merged['leaf_index'].unique()):
        sub = rt_clean[rt_clean['leaf_index'] == node]
        if len(sub) == 0:
            continue
        ax.scatter(sub[gold_col], sub[rt_col], s=8, alpha=0.5,
                   color=cmap(node / max(max_node, 1)), zorder=2,
                   label=f'leaf {node}' if node < 7 else None)
    ax.plot(ax_range, ax_range, 'k--', lw=0.7, alpha=0.6)
    ax.set_xlim(ax_range); ax.set_ylim(ax_range)
    ax.set_xlabel(f'Gold (Gaillard pipeline): {label}')
    ax.set_ylabel(f'E5 procedural round-trip: {label}')
    ax.set_aspect('equal')

axes[0].set_title('θ — leaf insertion angle')
axes[1].set_title('φ — leaf azimuth')
axes[0].legend(loc='upper left', frameon=True, fontsize=7, markerscale=1.5)
fig.suptitle(f'E5 — Filtered scatter (N_kept={len(rt_clean)}, gray=excluded)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'e5_scatter_filtered.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Side-by-side: unfiltered vs filtered vs E1b
e1b_path = OUT_DIR / 'e1b_per_leaf_phenosuite_vs_gold.csv'
rows_cmp = []

for label, sub_e5_all, sub_e5_filt in [
    ('Lower 4', rt[rt['leaf_index'] < 4], lower4_clean),
    ('All leaves', rt, rt_clean),
]:
    for exp_name, sub in [
        ('E5 all', sub_e5_all),
        ('E5 filtered', sub_e5_filt),
    ]:
        n = len(sub)
        if n < 2:
            continue
        r_t = pearsonr(sub['theta_gold'], sub['theta_rt'])[0]
        r_p = pearsonr(sub['phi_gold'], sub['phi_rt'])[0]
        rows_cmp.append({
            'Subset': label,
            'Experiment': exp_name,
            'N': n,
            'Median |Δθ|': sub['abs_dtheta'].median(),
            'RMSE θ': np.sqrt((sub['dtheta'] ** 2).mean()),
            'r(θ)': r_t,
            'R²(θ)': r_t ** 2,
            'Median |Δφ|': sub['abs_dphi'].median(),
            'RMSE φ': np.sqrt((sub['dphi'] ** 2).mean()),
            'r(φ)': r_p,
            'R²(φ)': r_p ** 2,
        })

    # Add E1b row if available
    if e1b_path.exists():
        e1b = pd.read_csv(e1b_path)
        sub_e1b = e1b[e1b['leaf_index'] < 4] if label == 'Lower 4' else e1b
        if len(sub_e1b) >= 2:
            r_t = pearsonr(sub_e1b['theta_gold'], sub_e1b['theta_ours'])[0]
            r_p = pearsonr(sub_e1b['phi_gold'], sub_e1b['phi_ours'])[0]
            rows_cmp.append({
                'Subset': label,
                'Experiment': 'E1b (spline)',
                'N': len(sub_e1b),
                'Median |Δθ|': sub_e1b['abs_dtheta'].median(),
                'RMSE θ': np.sqrt((sub_e1b['dtheta'] ** 2).mean()),
                'r(θ)': r_t,
                'R²(θ)': r_t ** 2,
                'Median |Δφ|': sub_e1b['abs_dphi'].median(),
                'RMSE φ': np.sqrt((sub_e1b['dphi'] ** 2).mean()),
                'r(φ)': r_p,
                'R²(φ)': r_p ** 2,
            })

cmp = pd.DataFrame(rows_cmp)
print(cmp.to_string(index=False, float_format='{:.2f}'.format))

## 6c. Visual comparison — skeleton vs procedural OBJ vs spline-override OBJ

Select a few representative plants at different fit-quality tiers (good / mediocre / poor) and export three files per plant for visual inspection in any 3D viewer:

1. **Skeleton point cloud** (`.obj`, vertices only) — the raw input.
2. **Procedural OBJ** — the mesh generated from the fitted `leaf_angle` + `droopiness` + `leaf_length` descriptor.
3. **Spline-override OBJ** — the mesh generated from the exact skeleton spline (`useCtrlOverrides=1`).

Comparing (1) vs (2) shows how well the procedural compression captures the real geometry.
Comparing (1) vs (3) shows how well the spline-override representation tracks the skeleton.

In [ ]:
from phenosuite.skeleton_to_descriptor import (
    skeleton_to_procedural_xml,
    skeleton_to_override_xml,
    skeleton_to_point_cloud_obj,
)
from phenosuite.wrapper import from_xml_to_obj, Maize

# Pick representative plants by median per-plant RMS error
plant_rms = fits.groupby('plant_id')['rms_error_cm'].median().sort_values()
n_plants = len(plant_rms)

picks = {
    'good':     plant_rms.index[n_plants // 10],       # 10th percentile
    'median':   plant_rms.index[n_plants // 2],         # 50th percentile
    'poor':     plant_rms.index[int(n_plants * 0.9)],   # 90th percentile
}

VIS_DIR = OUT_DIR / 'visual_comparison'
VIS_DIR.mkdir(parents=True, exist_ok=True)

VIS_STEM_RADIUS = 0.005   # 5 mm — ~1 voxel radius
VIS_LEAF_WIDTH  = 0.02   # 5 mm — thin ribbons for shape comparison

for tier, plant_id in picks.items():
    rms_val = plant_rms[plant_id]
    print(f'\n--- {tier} fit (median RMS = {rms_val:.2f} cm): {plant_id[:40]}... ---')
    sk = dl.load_skeleton(plant_id)
    prefix = VIS_DIR / f'{tier}'

    # 1. Skeleton point cloud
    skel_obj = str(prefix) + '_skeleton.obj'
    skeleton_to_point_cloud_obj(sk, skel_obj)
    print(f'  skeleton -> {Path(skel_obj).name}')

    # 2. Procedural descriptor -> OBJ (full mesh + stem-only)
    proc_xml = str(prefix) + '_procedural.xml'
    skeleton_to_procedural_xml(
        sk, proc_xml, stem_radius=VIS_STEM_RADIUS,
        spline_points=SPLINE_POINTS, leaf_width=VIS_LEAF_WIDTH,
    )
    proc_obj = str(prefix) + '_procedural.obj'
    ok = from_xml_to_obj(proc_xml, proc_obj)
    print(f'  procedural -> {Path(proc_obj).name} ({"OK" if ok else "FAILED"})')

    # 3. Spline-override descriptor -> OBJ
    ovr_xml = str(prefix) + '_override.xml'
    skeleton_to_override_xml(
        sk, ovr_xml, stem_radius=VIS_STEM_RADIUS,
        leaf_width=VIS_LEAF_WIDTH,
    )
    ovr_obj = str(prefix) + '_override.obj'
    ok = from_xml_to_obj(ovr_xml, ovr_obj)
    print(f'  override   -> {Path(ovr_obj).name} ({"OK" if ok else "FAILED"})')

print(f'\nAll visual comparison files saved to {VIS_DIR}/')
print(f'Stem radius = {VIS_STEM_RADIUS*1000:.0f} mm, leaf width = {VIS_LEAF_WIDTH*1000:.0f} mm')
print('Note: the "thick stem" appearance is from leaf-base geometry, not the stem mesh itself.')
print('Use wireframe mode in your viewer for the clearest shape comparison.')

## 7. Save outputs

In [ ]:
# Per-leaf fitted parameters
fits.to_csv(OUT_DIR / 'e5_fitted_params.csv', index=False)

# Per-leaf round-trip comparison (all + filter flag)
rt_merged.to_csv(OUT_DIR / 'e5_roundtrip_vs_gold.csv', index=False)

# Filtered subset only
rt_clean.to_csv(OUT_DIR / 'e5_roundtrip_filtered.csv', index=False)

# Per-node summary
node_rows = []
for node, grp in rt.groupby('leaf_index'):
    n = len(grp)
    row = {'leaf_index': node, 'n': n}
    if n >= 2:
        r_theta = pearsonr(grp['theta_gold'], grp['theta_rt'])[0]
        r_phi   = pearsonr(grp['phi_gold'],   grp['phi_rt'])[0]
        row['r_theta']  = r_theta
        row['R2_theta'] = r_theta ** 2
        row['r_phi']    = r_phi
        row['R2_phi']   = r_phi ** 2
    else:
        row['r_theta'] = row['R2_theta'] = row['r_phi'] = row['R2_phi'] = float('nan')
    row['median_abs_dtheta'] = grp['abs_dtheta'].median()
    row['median_abs_dphi']   = grp['abs_dphi'].median()
    row['rmse_theta'] = float(np.sqrt((grp['dtheta'] ** 2).mean()))
    row['rmse_phi']   = float(np.sqrt((grp['dphi'] ** 2).mean()))
    node_rows.append(row)

node_summary = pd.DataFrame(node_rows)
node_summary.to_csv(OUT_DIR / 'e5_per_node_summary.csv', index=False)

# Failures
if rt_failures:
    pd.DataFrame(rt_failures, columns=['plant_id', 'error_type', 'message']).to_csv(
        OUT_DIR / 'e5_failures.csv', index=False)

print('Saved:')
print(f'  outputs/e5_fitted_params.csv             ({len(fits)} leaves)')
print(f'  outputs/e5_roundtrip_vs_gold.csv         ({len(rt_merged)} comparisons, with filter flag)')
print(f'  outputs/e5_roundtrip_filtered.csv        ({len(rt_clean)} filtered comparisons)')
print(f'  outputs/e5_per_node_summary.csv          ({len(node_summary)} nodes)')
if rt_failures:
    print(f'  outputs/e5_failures.csv                  ({len(rt_failures)} plants)')
print(f'  outputs/visual_comparison/               (3 tiers × 3 files = skeleton + procedural + override OBJ)')
print(f'\nFigures:')
for fig_name in sorted(FIG_DIR.glob('e5_*.png')):
    print(f'  figures/{fig_name.name}')

## 8. Analysis

### 8.1 What E5 tests

E5 asks a stricter question than E1b. Instead of using spline overrides that closely follow the skeleton centerlines, each leaf is compressed into a small procedural parameter set: leaf length, insertion angle, droopiness, leaf curl, and azimuth. The descriptor is then regenerated and traits are measured again. The experiment therefore measures the cost of procedural compression on biologically relevant traits.

### 8.2 Fit quality

The inverse fit succeeds at population scale. Across 2,479 leaves from 324 plants (the same 8 plant-level failures seen in the other skeleton-based experiments), the RMS spline fitting error has a median of 2.43 cm, a mean of 2.91 cm, and a 90th percentile of 5.34 cm. For sorghum leaves that are typically 20–50 cm long, a 2.4 cm median positional error along the midrib indicates that the procedural model captures the dominant curvature pattern of most leaves while missing finer shape details.

The fitted parameter distributions are biologically plausible: leaf lengths span the expected range of 15–60 cm, insertion angles cover the full range from near-vertical lower leaves to near-horizontal upper leaves, and droopiness varies enough to distinguish erect from drooping architectures. These distributions are suitable candidates for E8/E9 population-level analyses, with fit residuals carried forward as a quality covariate.

### 8.3 Trait round-trip performance

On the lower-four-leaf subset (N = 1,285), the procedural round-trip gives median |Δθ| = 4.95 degrees and median |Δφ| = 7.41 degrees, with correlations r = 0.565 for theta and r = 0.643 for phi. Compared with the E1b spline-override baseline (median |Δθ| = 4.94 degrees, median |Δφ| = 5.59 degrees, r = 0.623 and r = 0.719), theta is essentially unchanged while phi shows a moderate penalty.

After filtering out low-voxel leaves and the worst 10% of procedural fits (keeping 88.1% of leaves), lower-four performance improves to r = 0.627 for theta with median |Δθ| = 5.02 degrees, and r = 0.652 for phi with median |Δφ| = 6.99 degrees. Filtered theta correlation (R² = 0.39) matches the E1b baseline (R² = 0.39). Filtered phi correlation (R² = 0.42) remains below the E1b baseline (R² = 0.52), indicating that procedural compression costs about 0.10 in R² for azimuth on the lower leaves.

### 8.4 Compression cost

The side-by-side comparison table quantifies the compression penalty:

| Subset | Experiment | R²(θ) | R²(φ) | Median \|Δθ\| | Median \|Δφ\| |
|--------|-----------|-------|-------|------------|------------|
| Lower 4 | E5 filtered | 0.39 | 0.42 | 5.02° | 6.99° |
| Lower 4 | E1b (spline) | 0.39 | 0.52 | 4.94° | 5.59° |
| All | E5 filtered | 0.40 | 0.27 | 6.54° | 10.21° |
| All | E1b (spline) | 0.36 | 0.35 | 5.05° | 8.56° |

Two patterns stand out. First, the theta compression cost is negligible: filtered E5 and E1b achieve indistinguishable R² for insertion angle on both subsets. The procedural model's curvature basis functions (the linear gravity term for droopiness, the cubic `4t(1-t)` term for leaf curl) are expressive enough to preserve the base tangent direction that determines theta. Second, the phi cost is concentrated on the lower-leaf subset — for all leaves, E5 filtered actually has a slightly higher R²(θ) than E1b (0.40 vs 0.36), likely because the quality filter preferentially removes noisy upper-leaf measurements.

### 8.5 Sources of residual error

Several mechanisms contribute to the remaining round-trip error. First, the procedural curvature model cannot represent all observed leaf centerline shapes — S-curves, abrupt tip bends, and asymmetric deformations fall outside the two-term basis. Second, trait extraction is sensitive to the base tangent at the leaf insertion point, so small differences in the first few millimetres of the fitted spline can produce several degrees of theta error even when the overall leaf shape looks similar. Third, phi remains vulnerable to the ±180-degree PCA sign ambiguity (17 of 324 plants required a residual flip, consistent with the 18 seen in E1b).

The fit-quality filter improves theta more clearly than phi, which supports this interpretation: geometric fit residual explains part of the insertion-angle error, while azimuth error is also affected by orientation conventions that the filter does not address.

### 8.6 Interpretation for the paper

E5 supports the claim that PhenoSuite can compress real sorghum skeletons into procedural descriptors while retaining useful trait information. The strongest statement is for lower-leaf theta, where filtered procedural performance matches the spline baseline. The more cautious statement is for phi: median error increases by about 1.4 degrees (from 5.59 to 6.99 degrees), and R² drops by 0.10 (from 0.52 to 0.42). This is a real but modest cost for reducing each leaf from an arbitrary-length spline to four numbers.

For downstream experiments, the fitted parameters in `e5_fitted_params.csv` are suitable for population-level analysis, with the caveat that `rms_error_cm` should be carried as a quality covariate and leaves with very high fit residuals should be flagged.